In [12]:
from pyspark.sql  import SparkSession

from pyspark.sql.types import (
StructType,  StructField,
IntegerType, FloatType, DecimalType, StringType, DoubleType, DateType, TimestampType
)

from pyspark.sql.functions import (
col, concat_ws, concat
)

In [13]:
spark = SparkSession.builder.appName("Cleaning").getOrCreate()

df = spark.read.csv("data/vgsales.csv",header=True, inferSchema=True)
df.printSchema()

df.show()

root
 |-- Rank: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Platform: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- NA_Sales: double (nullable = true)
 |-- EU_Sales: double (nullable = true)
 |-- JP_Sales: double (nullable = true)
 |-- Other_Sales: double (nullable = true)
 |-- Global_Sales: double (nullable = true)

+----+--------------------+--------+----+------------+--------------------+--------+--------+--------+-----------+------------+
|Rank|                Name|Platform|Year|       Genre|           Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|
+----+--------------------+--------+----+------------+--------------------+--------+--------+--------+-----------+------------+
|   1|          Wii Sports|     Wii|2006|      Sports|            Nintendo|   41.49|   29.02|    3.77|       8.46|       82.74|
|   2|   Super Mario Bros.|     NES|1985|    Pla

In [14]:
games_schema = StructType([
    StructField('Rank', IntegerType(), True),
    StructField('Name', StringType(), True),
    StructField('Platform', StringType(), True),
    StructField('Year', IntegerType(), True),
    StructField('Genre', StringType(), True),
    StructField('Publisher', StringType(), True),
    StructField('NA_Sales', DoubleType(), True),
])

games = spark.read.csv('data/vgsales.csv', header=True, schema=games_schema)
games.printSchema()
games.show()

root
 |-- Rank: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Platform: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- NA_Sales: double (nullable = true)

+----+--------------------+--------+----+------------+--------------------+--------+
|Rank|                Name|Platform|Year|       Genre|           Publisher|NA_Sales|
+----+--------------------+--------+----+------------+--------------------+--------+
|   1|          Wii Sports|     Wii|2006|      Sports|            Nintendo|   41.49|
|   2|   Super Mario Bros.|     NES|1985|    Platform|            Nintendo|   29.08|
|   3|      Mario Kart Wii|     Wii|2008|      Racing|            Nintendo|   15.85|
|   4|   Wii Sports Resort|     Wii|2009|      Sports|            Nintendo|   15.75|
|   5|Pokemon Red/Pokem...|      GB|1996|Role-Playing|            Nintendo|   11.27|
|   6|              Tetris|      GB|1989|     

In [15]:
games = games.select(games.Rank, col('Name'), games.Publisher, col('Year'), col('Genre'), col('Platform'), col('NA_Sales'))
games.printSchema()
games.show()

root
 |-- Rank: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Platform: string (nullable = true)
 |-- NA_Sales: double (nullable = true)

+----+--------------------+--------------------+----+------------+--------+--------+
|Rank|                Name|           Publisher|Year|       Genre|Platform|NA_Sales|
+----+--------------------+--------------------+----+------------+--------+--------+
|   1|          Wii Sports|            Nintendo|2006|      Sports|     Wii|   41.49|
|   2|   Super Mario Bros.|            Nintendo|1985|    Platform|     NES|   29.08|
|   3|      Mario Kart Wii|            Nintendo|2008|      Racing|     Wii|   15.85|
|   4|   Wii Sports Resort|            Nintendo|2009|      Sports|     Wii|   15.75|
|   5|Pokemon Red/Pokem...|            Nintendo|1996|Role-Playing|      GB|   11.27|
|   6|              Tetris|            Nintend

In [16]:
games.filter(games.Rank<=10).show()

+----+--------------------+---------+----+------------+--------+--------+
|Rank|                Name|Publisher|Year|       Genre|Platform|NA_Sales|
+----+--------------------+---------+----+------------+--------+--------+
|   1|          Wii Sports| Nintendo|2006|      Sports|     Wii|   41.49|
|   2|   Super Mario Bros.| Nintendo|1985|    Platform|     NES|   29.08|
|   3|      Mario Kart Wii| Nintendo|2008|      Racing|     Wii|   15.85|
|   4|   Wii Sports Resort| Nintendo|2009|      Sports|     Wii|   15.75|
|   5|Pokemon Red/Pokem...| Nintendo|1996|Role-Playing|      GB|   11.27|
|   6|              Tetris| Nintendo|1989|      Puzzle|      GB|    23.2|
|   7|New Super Mario B...| Nintendo|2006|    Platform|      DS|   11.38|
|   8|            Wii Play| Nintendo|2006|        Misc|     Wii|   14.03|
|   9|New Super Mario B...| Nintendo|2009|    Platform|     Wii|   14.59|
|  10|           Duck Hunt| Nintendo|1984|     Shooter|     NES|   26.93|
+----+--------------------+---------+-

In [17]:
games.filter(col('Publisher')!="Nintendo").show()

+----+--------------------+--------------------+----+-------+--------+--------+
|Rank|                Name|           Publisher|Year|  Genre|Platform|NA_Sales|
+----+--------------------+--------------------+----+-------+--------+--------+
|  16|  Kinect Adventures!|Microsoft Game St...|2010|   Misc|    X360|   14.97|
|  17|  Grand Theft Auto V|Take-Two Interactive|2013| Action|     PS3|    7.01|
|  18|Grand Theft Auto:...|Take-Two Interactive|2004| Action|     PS2|    9.43|
|  24|  Grand Theft Auto V|Take-Two Interactive|2013| Action|    X360|    9.63|
|  25|Grand Theft Auto:...|Take-Two Interactive|2002| Action|     PS2|    8.41|
|  29|Gran Turismo 3: A...|Sony Computer Ent...|2001| Racing|     PS2|    6.85|
|  30|Call of Duty: Mod...|          Activision|2011|Shooter|    X360|    9.03|
|  32|Call of Duty: Bla...|          Activision|2010|Shooter|    X360|    9.67|
|  34|Call of Duty: Bla...|          Activision|2015|Shooter|     PS4|    5.77|
|  35|Call of Duty: Bla...|          Act

In [18]:
games.filter(col('Year')>=2010).show()

+----+--------------------+--------------------+----+------------+--------+--------+
|Rank|                Name|           Publisher|Year|       Genre|Platform|NA_Sales|
+----+--------------------+--------------------+----+------------+--------+--------+
|  16|  Kinect Adventures!|Microsoft Game St...|2010|        Misc|    X360|   14.97|
|  17|  Grand Theft Auto V|Take-Two Interactive|2013|      Action|     PS3|    7.01|
|  24|  Grand Theft Auto V|Take-Two Interactive|2013|      Action|    X360|    9.63|
|  27|Pokemon Black/Pok...|            Nintendo|2010|Role-Playing|      DS|    5.57|
|  30|Call of Duty: Mod...|          Activision|2011|     Shooter|    X360|    9.03|
|  32|Call of Duty: Bla...|          Activision|2010|     Shooter|    X360|    9.67|
|  33| Pokemon X/Pokemon Y|            Nintendo|2013|Role-Playing|     3DS|    5.17|
|  34|Call of Duty: Bla...|          Activision|2015|     Shooter|     PS4|    5.77|
|  35|Call of Duty: Bla...|          Activision|2012|     Shooter

In [19]:
games.head(5)
games.tail(5)

[Row(Rank=16596, Name='Woody Woodpecker in Crazy Castle 5', Publisher='Kemco', Year=2002, Genre='Platform', Platform='GBA', NA_Sales=0.01),
 Row(Rank=16597, Name='Men in Black II: Alien Escape', Publisher='Infogrames', Year=2003, Genre='Shooter', Platform='GC', NA_Sales=0.01),
 Row(Rank=16598, Name='SCORE International Baja 1000: The Official Game', Publisher='Activision', Year=2008, Genre='Racing', Platform='PS2', NA_Sales=0.0),
 Row(Rank=16599, Name='Know How 2', Publisher='7G//AMES', Year=2010, Genre='Puzzle', Platform='DS', NA_Sales=0.0),
 Row(Rank=16600, Name='Spirits & Spells', Publisher='Wanadoo', Year=2003, Genre='Platform', Platform='GBA', NA_Sales=0.01)]

In [20]:
games.where(games.NA_Sales == 0).show()

+----+--------------------+--------------------+----+------------+--------+--------+
|Rank|                Name|           Publisher|Year|       Genre|Platform|NA_Sales|
+----+--------------------+--------------------+----+------------+--------+--------+
| 215|Monster Hunter Fr...|              Capcom|2010|Role-Playing|     PSP|     0.0|
| 339|   Friend Collection|            Nintendo|2009|        Misc|      DS|     0.0|
| 384|    Monster Hunter 4|              Capcom|2013|Role-Playing|     3DS|     0.0|
| 403|English Training:...|            Nintendo|2006|        Misc|      DS|     0.0|
| 427|Dragon Quest VI: ...|    Enix Corporation|1995|Role-Playing|    SNES|     0.0|
| 532|Dragon Quest V: T...|    Enix Corporation|1992|Role-Playing|    SNES|     0.0|
| 562|Yokai Watch 2 Shi...|             Level 5|2014|Role-Playing|     3DS|     0.0|
| 574|Super Mario Bros....|            Nintendo|1986|    Platform|     NES|     0.0|
| 630|     Final Fantasy V|          SquareSoft|1992|Role-Playing

In [21]:
spark.stop()